# Model 1 — Crop Recommendation (Kerala)

This notebook trains a **Kerala-augmented crop recommender** on the Kaggle *Crop Recommendation Dataset* (Atharva Ingle) and adds Kerala-specific synthetic categorical context:

- `district` (Kerala districts)
- `soil_type` (Laterite/Alluvial/Sandy_Loam/Red_Loam/Forest)
- `season` (Kharif/Rabi/Summer)
- `water_source` (Rainfall/Canal/Borewell/Pond/River)

Outputs a **top-5** crop shortlist with a suitability score and Kerala-friendly reasoning.


In [ ]:
# Run this first in every notebook
!pip -q install scikit-learn xgboost tensorflow pandas numpy matplotlib
!pip -q install seaborn plotly joblib opencv-python Pillow
!pip -q install kaggle

from google.colab import drive
drive.mount('/content/drive')

# Set paths (you can change these)
MODEL_SAVE_PATH = '/content/drive/MyDrive/krishi_mitra_models/'
DATA_PATH = '/content/drive/MyDrive/krishi_mitra_data/'

import os
os.makedirs(MODEL_SAVE_PATH, exist_ok=True)
os.makedirs(DATA_PATH, exist_ok=True)


## 1) Download dataset (Kaggle)

Dataset: `atharvaingle/crop-recommendation-dataset`

### Colab steps
1. Upload your `kaggle.json` to Colab (left sidebar → Files)
2. Run the cell below

After download, this notebook expects `Crop_recommendation.csv`.


In [ ]:
import os
from pathlib import Path

kaggle_dir = Path('/root/.kaggle')
kaggle_dir.mkdir(parents=True, exist_ok=True)

# If you uploaded kaggle.json into Colab Files, copy it:
if Path('kaggle.json').exists():
    !cp kaggle.json /root/.kaggle/kaggle.json
    !chmod 600 /root/.kaggle/kaggle.json

raw_dir = Path(DATA_PATH) / 'raw'
raw_dir.mkdir(parents=True, exist_ok=True)

# Download (safe to rerun)
!kaggle datasets download -d atharvaingle/crop-recommendation-dataset -p "{raw_dir}" -q
!unzip -o "{raw_dir}/crop-recommendation-dataset.zip" -d "{raw_dir}" > /dev/null

raw_csv = raw_dir / 'Crop_recommendation.csv'
print('CSV exists:', raw_csv.exists(), raw_csv)


## 2) Load + Kerala augmentation

The Kaggle dataset has only numeric agronomic/climate features.

Kerala-specific features are **synthetic** (district/soil/season/water_source) and are generated in a *plausible* way using district climate priors (used only to create a useful, Kerala-aware model baseline).


In [ ]:
import numpy as np
import pandas as pd

df = pd.read_csv(raw_csv)
df.columns = [c.strip() for c in df.columns]
df.rename(columns={'label': 'crop'}, inplace=True)
df.head()

In [ ]:
KERALA_DISTRICTS = [
    'Palakkad','Thrissur','Wayanad','Malappuram','Kannur',
    'Kozhikode','Idukki','Ernakulam','Alappuzha','Kottayam'
]
SOIL_TYPES = ['Laterite','Alluvial','Sandy_Loam','Red_Loam','Forest']
SEASONS = ['Kharif','Rabi','Summer']
WATER_SOURCES = ['Rainfall','Canal','Borewell','Pond','River']

DISTRICT_PRIORS = {
    'Palakkad': {'temp': 29.0, 'rain': 180.0, 'hum': 78.0},
    'Thrissur': {'temp': 28.0, 'rain': 210.0, 'hum': 82.0},
    'Wayanad': {'temp': 24.5, 'rain': 260.0, 'hum': 88.0},
    'Malappuram': {'temp': 27.5, 'rain': 230.0, 'hum': 84.0},
    'Kannur': {'temp': 27.0, 'rain': 240.0, 'hum': 86.0},
    'Kozhikode': {'temp': 27.0, 'rain': 245.0, 'hum': 87.0},
    'Idukki': {'temp': 23.5, 'rain': 250.0, 'hum': 86.0},
    'Ernakulam': {'temp': 28.0, 'rain': 220.0, 'hum': 85.0},
    'Alappuzha': {'temp': 28.2, 'rain': 235.0, 'hum': 88.0},
    'Kottayam': {'temp': 27.2, 'rain': 240.0, 'hum': 87.0},
}

KERALA_CROPS = [
    'Paddy','Coconut','Banana','Tapioca','Rubber','Black_Pepper','Cardamom',
    'Ginger','Turmeric','Coffee','Tea','Arecanut','Cashew','Mango','Jackfruit',
    'Maize','Sugarcane','Bitter_Gourd','Okra','Brinjal'
]

rng = np.random.default_rng(42)

def sample_season(month):
    # Kerala-like heuristic: monsoon ~ Jun-Sep; rabi ~ Oct-Feb; summer ~ Mar-May
    if month in [6,7,8,9]:
        return 'Kharif'
    if month in [10,11,12,1,2]:
        return 'Rabi'
    return 'Summer'

def assign_kerala_context(n):
    districts = rng.choice(KERALA_DISTRICTS, size=n)
    soils = rng.choice(SOIL_TYPES, size=n, p=[0.38, 0.22, 0.12, 0.18, 0.10])
    months = rng.integers(1, 13, size=n)
    seasons = np.array([sample_season(m) for m in months])
    # water_source correlated with season
    water = []
    for s in seasons:
        if s == 'Kharif':
            water.append(rng.choice(WATER_SOURCES, p=[0.55,0.2,0.05,0.1,0.1]))
        elif s == 'Rabi':
            water.append(rng.choice(WATER_SOURCES, p=[0.2,0.35,0.15,0.15,0.15]))
        else:
            water.append(rng.choice(WATER_SOURCES, p=[0.15,0.25,0.25,0.2,0.15]))
    return districts, soils, seasons, np.array(water)

district, soil_type, season, water_source = assign_kerala_context(len(df))
df['district'] = district
df['soil_type'] = soil_type
df['season'] = season
df['water_source'] = water_source

# Slightly nudge climate features toward district priors (keeps original signal, adds Kerala context)
temp_prior = np.array([DISTRICT_PRIORS[d]['temp'] for d in df['district']])
rain_prior = np.array([DISTRICT_PRIORS[d]['rain'] for d in df['district']])
hum_prior = np.array([DISTRICT_PRIORS[d]['hum'] for d in df['district']])

df['temperature'] = 0.85 * df['temperature'] + 0.15 * temp_prior
df['rainfall'] = 0.85 * df['rainfall'] + 0.15 * rain_prior
df['humidity'] = 0.85 * df['humidity'] + 0.15 * hum_prior

df[['district','soil_type','season','water_source','temperature','humidity','rainfall']].head()

## 3) Feature engineering

- Categorical: one-hot encode `district`, `soil_type`, `season`, `water_source`
- Numeric: scale `N`, `P`, `K`, `temperature`, `humidity`, `ph`, `rainfall`


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

X = df[['N','P','K','temperature','humidity','ph','rainfall','district','soil_type','season','water_source']].copy()
y = df['crop'].astype(str).copy()

label_encoder = LabelEncoder()
y_enc = label_encoder.fit_transform(y)

num_cols = ['N','P','K','temperature','humidity','ph','rainfall']
cat_cols = ['district','soil_type','season','water_source']

preprocess = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
    ],
    remainder='drop'
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.2, random_state=42, stratify=y_enc
)


## 4) Train + compare models

Primary: **Random Forest** (strong baseline for mixed-tabular with non-linearities)

Also compare: XGBoost, KNN, Naive Bayes


In [ ]:
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB

models = {
    'RandomForest': RandomForestClassifier(
        n_estimators=400, random_state=42, n_jobs=-1, class_weight='balanced_subsample'
    ),
    'KNN': KNeighborsClassifier(n_neighbors=15),
    'NaiveBayes': GaussianNB(),
}

try:
    from xgboost import XGBClassifier
    models['XGBoost'] = XGBClassifier(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=8,
        subsample=0.9,
        colsample_bytree=0.9,
        reg_lambda=1.0,
        objective='multi:softprob',
        eval_metric='mlogloss',
        random_state=42,
        n_jobs=-1,
    )
except Exception as e:
    print('XGBoost unavailable:', e)

results = []
fitted = {}
for name, clf in models.items():
    pipe = Pipeline([('pre', preprocess), ('clf', clf)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    acc = accuracy_score(y_test, pred)
    results.append((name, acc))
    fitted[name] = pipe

pd.DataFrame(results, columns=['model','accuracy']).sort_values('accuracy', ascending=False)

## 5) Hyperparameter tuning (GridSearchCV) — Random Forest

We tune Random Forest as the default production choice because it is robust and easy to serialize.


In [ ]:
from sklearn.model_selection import GridSearchCV

rf_pipe = Pipeline([
    ('pre', preprocess),
    ('clf', RandomForestClassifier(random_state=42, n_jobs=-1, class_weight='balanced_subsample'))
])

param_grid = {
    'clf__n_estimators': [300, 500],
    'clf__max_depth': [None, 18, 26],
    'clf__min_samples_split': [2, 5],
    'clf__min_samples_leaf': [1, 2],
    'clf__max_features': ['sqrt', 0.6],
}

grid = GridSearchCV(
    rf_pipe,
    param_grid=param_grid,
    cv=3,
    n_jobs=-1,
    verbose=1,
    scoring='accuracy'
)
grid.fit(X_train, y_train)

print('Best params:', grid.best_params_)
print('Best CV accuracy:', grid.best_score_)

best_model = grid.best_estimator_


## 6) Evaluation

We report accuracy and a confusion matrix. (For multi-class datasets, macro averages can be added if needed.)


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

y_pred = best_model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print('Test accuracy:', acc)
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_[:]))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, cmap='Blues', cbar=True)
plt.title('Confusion Matrix (encoded classes)')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.show()

## 7) Save model artifacts

Saves to:
- local project path: `ml_models/1_crop_recommender/model/`
- and also to Drive folder `MODEL_SAVE_PATH` for persistence in Colab


In [ ]:
import joblib
from pathlib import Path

# Local save (in Colab this path may not exist unless you upload the repo)
local_model_dir = Path('ml_models/1_crop_recommender/model')
local_model_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(best_model, local_model_dir / 'crop_model.pkl')
joblib.dump(label_encoder, local_model_dir / 'label_encoder.pkl')

# Drive save
drive_dir = Path(MODEL_SAVE_PATH) / 'crop_recommender'
drive_dir.mkdir(parents=True, exist_ok=True)
joblib.dump(best_model, drive_dir / 'crop_model.pkl')
joblib.dump(label_encoder, drive_dir / 'label_encoder.pkl')

print('Saved to local:', local_model_dir)
print('Saved to drive:', drive_dir)

## 8) `predict()` demo (Top-5 crops)

We compute `suitability_score` from `predict_proba` and attach lightweight Kerala-oriented reasoning.


In [ ]:
def _season_hint(crop: str) -> str:
    mapping = {
        'Paddy': 'Kharif',
        'Banana': 'Year-round (avoid peak flood weeks)',
        'Coconut': 'Year-round',
        'Black_Pepper': 'Kharif/Rabi (with drainage)',
        'Ginger': 'Kharif',
        'Turmeric': 'Kharif',
        'Tapioca': 'Rabi/Summer',
        'Rubber': 'Monsoon planting',
    }
    return mapping.get(crop, 'Season depends on local practice')

def _avg_yield_hint(crop: str) -> str:
    # Simple, human-friendly placeholder (tons/ha). Replace with Kerala stats if you have them.
    yields = {
        'Paddy': '3–5 t/ha',
        'Banana': '25–40 t/ha',
        'Coconut': '8,000–12,000 nuts/ha',
        'Tapioca': '20–30 t/ha',
        'Ginger': '10–15 t/ha (green)',
        'Turmeric': '20–30 t/ha (green)',
        'Black_Pepper': '0.3–0.8 t/ha (dry)',
        'Rubber': '1.2–1.8 t/ha (dry rubber)',
    }
    return yields.get(crop, 'Varies by variety & management')

def _reason(inp: dict, crop: str, score: float) -> str:
    reasons = []
    if inp.get('season') == 'Kharif' and inp.get('rainfall', 0) > 150:
        reasons.append('Good monsoon moisture')
    if inp.get('soil_type') in ['Laterite','Red_Loam'] and crop in ['Cashew','Tapioca','Mango','Jackfruit']:
        reasons.append('Matches laterite/red loam adaptation')
    if inp.get('humidity', 0) > 80 and crop in ['Paddy','Banana','Coconut','Black_Pepper']:
        reasons.append('High humidity supports this crop (ensure disease management)')
    if crop == 'Paddy' and inp.get('water_source') in ['Canal','River','Pond']:
        reasons.append('Reliable water source suits paddy')
    if not reasons:
        reasons.append('Overall feature match to training patterns')
    return f"Score {score:.2f}. " + '; '.join(reasons)

def predict(farmer_input: dict, top_k: int = 5):
    required = ['district','soil_type','N','P','K','pH','rainfall','temperature','humidity','season','water_source']
    missing = [k for k in required if k not in farmer_input]
    if missing:
        raise ValueError(f"Missing keys: {missing}")

    row = {
        'district': farmer_input['district'],
        'soil_type': farmer_input['soil_type'],
        'N': farmer_input['N'],
        'P': farmer_input['P'],
        'K': farmer_input['K'],
        'ph': farmer_input['pH'],
        'rainfall': farmer_input['rainfall'],
        'temperature': farmer_input['temperature'],
        'humidity': farmer_input['humidity'],
        'season': farmer_input['season'],
        'water_source': farmer_input['water_source'],
    }

    X_one = pd.DataFrame([row])
    proba = best_model.predict_proba(X_one)[0]
    idx = np.argsort(proba)[::-1][:top_k]

    out = []
    for i in idx:
        crop = label_encoder.inverse_transform([i])[0]
        score = float(proba[i])
        out.append({
            'crop': crop,
            'suitability_score': score,
            'reason': _reason(farmer_input, crop, score),
            'best_season': _season_hint(crop),
            'avg_yield': _avg_yield_hint(crop),
        })
    return out

test_farmer = {
    'district': 'Palakkad',
    'soil_type': 'Laterite',
    'N': 90, 'P': 42, 'K': 43,
    'pH': 6.5,
    'rainfall': 200,
    'temperature': 28,
    'humidity': 82,
    'season': 'Kharif',
    'water_source': 'Canal',
}

preds = predict(test_farmer)
pd.DataFrame(preds)